# EnsembleAgeHumanMouse

## Index
1. [Instantiate model class](#Instantiate-model-class)
2. [Define clock metadata](#Define-clock-metadata)
3. [Download clock dependencies](#Download-clock-dependencies)
4. [Load features](#Load-features)
5. [Load weights into base model](#Load-weights-into-base-model)
6. [Load reference values](#Load-reference-values)
7. [Load preprocess and postprocess objects](#Load-preprocess-and-postprocess-objects)
8. [Check all clock parameters](#Check-all-clock-parameters)
9. [Normal feature ranges](#Normal-feature-ranges)
10. [Basic test](#Basic-test)
11. [Save torch model](#Save-torch-model)
12. [Clear directory](#Clear-directory)

Let's first import some packages:

In [1]:
import os
import inspect
import shutil
import json
import math
import torch
import pandas as pd
import pyaging as pya

## Instantiate model class

In [2]:
def print_entire_class(cls):
    source = inspect.getsource(cls)
    print(source)

print_entire_class(pya.models.EnsembleAgeHumanMouse)

class EnsembleAgeHumanMouse(LinearReferenceClock):
    pass



In [3]:
model = pya.models.EnsembleAgeHumanMouse()

## Define clock metadata

In [4]:
model.metadata["clock_name"] = "ensembleagehumanmouse"
model.metadata["data_type"] = "DNA methylation"  # Paper: EnsembleAge integrates predictions from multiple penalized models.
model.metadata["species"] = "Homo sapiens and Mus musculus"  # Paper: A merged human-mouse dataset enabled cross-species predictions.
model.metadata["year"] = 2025
model.metadata["approved_by_author"] = "⌛"
model.metadata["citation"] = "Haghani, A. et al. EnsembleAge: enhancing epigenetic age assessment with a multi-clock framework. GeroScience 48, 2873-2886 (2025)."
model.metadata["doi"] = "https://doi.org/10.1007/s11357-025-01808-1"
model.metadata["notes"] = "Cross-species static EnsembleAge model trained on merged human and mouse methylation data; age is normalized by species maximum lifespan."
model.metadata["research_only"] = None
model.metadata["tissue"] = ["multi-tissue", "whole blood"]  # Paper: The human dataset consisted of 81 blood samples; mouse data covered various tissues.
model.metadata["predicts"] = ["relative age"]  # Paper: EnsembleAge integrates predictions from multiple penalized models.
model.metadata["training_target"] = ["relative age"]  # Paper: Age was normalized by maximum species lifespan.
model.metadata["unit"] = ["relative age"]  # Paper: HumanMouse clocks predict relative age (age/maximum species lifespan).
model.metadata["model_type"] = "elastic net regression"  # Paper: The clocks were trained using ridge, lasso, and elastic net regression.
model.metadata["platform"] = ["mammalian methylation array"]  # Paper: The DNA methylation data used in this study were generated using either the Mammal40k or Mammal320k BeadChip platforms.
model.metadata["population"] = "humans and mice"  # Paper: Mouse and human methylation data were combined for cross-species predictions.
model.metadata["journal"] = "GeroScience"
model.metadata["last_author"] = "Steve Horvath"
model.metadata["n_features"] = 100
model.metadata["citations"] = 3
model.metadata["citations_date"] = "2026-07-05"


## Download clock dependencies

In [5]:
supplementary_url = "https://raw.githubusercontent.com/Duzhaozhen/OmniAge/c10fbe8cb92957520fbff1d55ae1def0691252e5/OmniAgePy/src/omniage/data/EnsembleAge/EnsembleAge_HumanMouse_HumanMouse_coefs.csv"
supplementary_file_name = "coefficients.csv"
os.system(f"curl -sL -o {supplementary_file_name} {supplementary_url}")

0

## Load features

In [6]:
df = pd.read_csv('coefficients.csv')
if str(df.columns[0]).startswith('Unnamed'):
    df = df.iloc[:, 1:]
mask = df['probe'].astype(str).str.lower().isin(['intercept', '(intercept)'])
intercept_value = float(df.loc[mask, 'coef'].iloc[0]) if mask.any() else 0.0
coef_df = df.loc[~mask].reset_index(drop=True)
model.features = coef_df['probe'].tolist()

## Load weights into base model

In [7]:
weights = torch.tensor(coef_df['coef'].tolist()).unsqueeze(0).float()
intercept = torch.tensor([intercept_value]).float()

In [8]:
base_model = pya.models.LinearModel(input_dim=len(model.features))

base_model.linear.weight.data = weights.float()
base_model.linear.bias.data = intercept.float()

model.base_model = base_model

## Load reference values

In [9]:
model.reference_values = None

## Load preprocess and postprocess objects

In [10]:
model.preprocess_name = None
model.preprocess_dependencies = None

In [11]:
model.postprocess_name = None
model.postprocess_dependencies = None

## Check all clock parameters

In [12]:
pya.utils.print_model_details(model)


%==================================== Model Details ====================================%
Model Attributes:

training: True
metadata: {'approved_by_author': '⌛',
 'citation': 'Haghani, Amin, et al. "EnsembleAge: an ensemble of epigenetic '
             'clocks for robust age estimation." GeroScience (2025).',
 'clock_name': 'ensembleagehumanmouse',
 'data_type': 'methylation',
 'doi': 'https://doi.org/10.1007/s11357-025-01808-1',
 'notes': None,
 'research_only': None,
 'species': 'Homo sapiens',
 'version': None,
 'year': 2025}
reference_values: None
preprocess_name: None
preprocess_dependencies: None
postprocess_name: None
postprocess_dependencies: None
features: ['cg00001364', 'cg00001582', 'cg00003994', 'cg00005112', 'cg00051782', 'cg00060304', 'cg00066554', 'cg00067884', 'cg00073543', 'cg00079224', 'cg00084577', 'cg00091964', 'cg00096922', 'cg00109076', 'cg00109300', 'cg00116234', 'cg00146676', 'cg00158333', 'cg00159243', 'cg00167491', 'cg00187380', 'cg00211337', 'cg00216659', 'c

base_model.linear.bias: tensor([-0.5651])

%==================================== Model Details ====================================%



## Normal feature ranges

In [ ]:
# Units and plausibility ranges come from the package registry, keyed by feature name.
feature_ranges = pya.utils.resolve_feature_ranges(model.features, model.metadata["data_type"])
model.feature_units = [record["unit"] for record in feature_ranges]
pd.DataFrame.from_records(feature_ranges).head()

## Basic test

In [ ]:
# Exercise the clock with values in the middle of each feature's expected range.
records = pya.utils.resolve_feature_ranges(model.features, model.metadata["data_type"])
midpoints = [
    (record["low"] + record["high"]) / 2 if math.isfinite(record["high"]) else max(record["low"], 1.0)
    for record in records
]
input = torch.tensor([midpoints] * 10, dtype=torch.float64)
model.eval()
model.to(torch.float64)
pred = model(input)
pred

## Save torch model

In [14]:
torch.save(model, f"../weights/{model.metadata['clock_name']}.pt")

## Clear directory
<a id="10"></a>

In [15]:
# Function to remove a folder and all its contents
def remove_folder(path):
    try:
        shutil.rmtree(path)
        print(f"Deleted folder: {path}")
    except Exception as e:
        print(f"Error deleting folder {path}: {e}")

# Get a list of all files and folders in the current directory
all_items = os.listdir('.')

# Loop through the items
for item in all_items:
    # Check if it's a file and does not end with .ipynb
    if os.path.isfile(item) and not item.endswith('.ipynb'):
        os.remove(item)
        print(f"Deleted file: {item}")
    # Check if it's a folder
    elif os.path.isdir(item):
        remove_folder(item)

Deleted file: coefficients.csv
